# Computational modelling of hiPSC-derived neuronal networks
## Part 2: Simulation-Based Inference
**Neuroengineering Summer School 'Massimo Grattarola', Camogli 2026**

---

In this notebook we use **Neural Posterior Estimation (NPE)** to infer the
free parameters of the neuronal network model from the experimental recordings.

Instead of running a simulation for every possible parameter combination and
checking which fits best, NPE trains a neural network to directly learn the
mapping from summary features to a probability distribution over parameters --
the **posterior** $p(\\theta \\mid x_{obs})$. Once trained, inference is
instant: you feed in the observed features and the network returns the
posterior immediately.

A posterior has been pre-trained for you on 2000 simulations.

**Structure:**
1. Setup of notebook dependencies
2. Features of the experimental data
3. Inference with the pre-trained posterior
4. Exercise A: how does feature choice affect inference?
5. Exercise B (optional): what do the features actually encode?
6. Optional: train your own posterior

## 1. Setup

In [ ]:
!pip install brian2 sbi -q
!wget -q https://raw.githubusercontent.com/ninontwik/NeuroEngSummerSchool2026/main/data/Healthycontrol_MEArecording.csv
!wget -q https://raw.githubusercontent.com/ninontwik/NeuroEngSummerSchool2026/main/data/KCNQ2patient_MEArecording.csv
!wget -q https://raw.githubusercontent.com/ninontwik/NeuroEngSummerSchool2026/main/data/inference.pkl
!pip install gdown -q
import gdown
gdown.download(id='1f2PxVr7lvR8gOC5VXO0YirVyuVsEY4PD', output='parameters.npy', quiet=True)
gdown.download(id='1OhRY_U_IJU8jHylcr2aBtBubH1rS6oUO',   output='features.npy',   quiet=True)
gdown.download(id='1ZDQZ8L08-AiWnVV2r0VUqYZh137sQZB1',      output='valid.npy',      quiet=True)
print('Setup complete.')

In [ ]:
import builtins
import numpy as np
import matplotlib.pyplot as plt
import torch
import pickle
from sbi import analysis
from sbi.utils import BoxUniform

# Prior bounds for a uniform prior distribution. Assume these are based on biological ranges.
PRIOR_LOW  = np.array([  3.0,   500.0,  0.5])
PRIOR_HIGH = np.array([ 60.0,  4000.0,  4.0])

prior_limits = [
    [PRIOR_LOW[0],  PRIOR_HIGH[0]],
    [PRIOR_LOW[1],  PRIOR_HIGH[1]],
    [PRIOR_LOW[2],  PRIOR_HIGH[2]],
]
param_labels  = ['b_adapt (pA)', 'Poisson rate (Hz)', 'g_syn (nS)']
feature_names = [
    'mean_firing_rate',
    'n_active_units',
    'cv_firing_rate',
    'poprate_variance',
    'early_late_ratio',
    'mean_pairwise_corr',
]
REC_DURATION = 180.0

In [ ]:
# compute_features -- same function as in notebook 1.
# Defined here so this notebook is self-contained.

def compute_features(spike_data, n_units, sim_duration, bin_size=0.05):
    bin_edges = np.arange(0, sim_duration + bin_size, bin_size)
    n_bins = builtins.len(bin_edges) - 1
    unit_rates = np.zeros(n_units)
    for i in builtins.range(n_units):
        unit_rates[i] = int(np.sum(spike_data[:, 0] == i)) / sim_duration
    feat_mean_fr = np.mean(unit_rates)
    feat_cv_fr = np.std(unit_rates) / (np.mean(unit_rates) + 1e-9)
    rate_matrix = np.zeros((n_units, n_bins))
    for i in builtins.range(n_units):
        st = spike_data[spike_data[:, 0] == i, 1]
        if builtins.len(st) > 0:
            counts, _ = np.histogram(st, bins=bin_edges)
            rate_matrix[i] = counts / bin_size
    pop_rate = np.sum(rate_matrix, axis=0)
    feat_poprate_var = np.var(pop_rate)
    active_mask = unit_rates > 0.05
    active_rates = rate_matrix[active_mask]
    if active_rates.shape[0] > 1:
        corr_matrix = np.corrcoef(active_rates)
        upper_tri = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
        feat_mean_corr = float(np.mean(upper_tri))
    else:
        feat_mean_corr = 0.0
    half = n_bins // 2
    feat_early_late = np.mean(pop_rate[:half]) / (np.mean(pop_rate[half:]) + 1e-9)
    feat_n_active = int(np.sum(unit_rates > 0))
    return {
        'mean_firing_rate'   : float(feat_mean_fr),
        'n_active_units'     : float(feat_n_active),
        'cv_firing_rate'     : float(feat_cv_fr),
        'poprate_variance'   : float(feat_poprate_var),
        'early_late_ratio'   : float(feat_early_late),
        'mean_pairwise_corr' : feat_mean_corr,
    }

## 2. Features of the experimental data

Compute the summary features from both recordings.
These feature vectors are what the NDE conditions on to produce
a posterior over the three free parameters.

In [ ]:
spike_data_healthy = np.loadtxt('Healtycontrol_MEArecording.csv', delimiter=',', skiprows=1)
spike_data_disease = np.loadtxt('KCNQ2patient_MEArecording.csv', delimiter=',', skiprows=1)

n_units_healthy = int(spike_data_healthy[:, 0].max()) + 1
n_units_disease = int(spike_data_disease[:, 0].max()) + 1

features_healthy = compute_features(spike_data_healthy, n_units_healthy, REC_DURATION)
features_disease = compute_features(spike_data_disease, n_units_disease, REC_DURATION)

print(f'  {"Feature":<25}  {"Healthy":>10}  {"Disease":>10}')
print(f'  {"-"*25}  {"-"*10}  {"-"*10}')
for name in feature_names:
    print(f'  {name:<25}  {features_healthy[name]:>10.3f}  {features_disease[name]:>10.3f}')

In [ ]:
x_healthy = torch.tensor(
    [features_healthy[f] for f in feature_names], dtype=torch.float32
)
x_disease = torch.tensor(
    [features_disease[f] for f in feature_names], dtype=torch.float32
)

## 3. Inference with the pre-trained posterior

We load the pre-trained NPE posterior and condition it on the feature vectors
of each recording. This gives us a probability distribution over the three
free parameters -- the **posterior** $p(\\theta \\mid x_{obs})$.

We draw 2000 samples from the posterior to visualise it, and also compute
the **MAP estimate** (Maximum A Posteriori) -- the single parameter set
that has the highest posterior probability. Think of this as the best single
guess for the parameters, given the observed features.

In [ ]:
with open('inference.pkl', 'rb') as f:
    inference_loaded = pickle.load(f)
posterior = inference_loaded.build_posterior()
print('Posterior loaded.')

In [ ]:
# Sample from the posterior conditioned on the healthy recording.
# posterior.sample((N,), x=x_obs) draws N parameter sets from p(theta | x_obs).
samples_healthy = posterior.sample((2000,), x=x_healthy, show_progress_bars=False)

# Compute the MAP estimate -- the mode of the posterior.
# set_default_x() tells the posterior which observation to use for .map().
posterior.set_default_x(x_healthy)
map_healthy = posterior.map(show_progress_bars=False).squeeze()

print('MAP estimate (healthy):')
for j, name in builtins.enumerate(param_labels):
    print(f'  {name:<25} {float(map_healthy[j]):.3f}')

In [ ]:
samples_disease = posterior.sample((2000,), x=x_disease, show_progress_bars=False)
posterior.set_default_x(x_disease)
map_disease = posterior.map(show_progress_bars=False).squeeze()

print('MAP estimate (disease):')
for j, name in builtins.enumerate(param_labels):
    print(f'  {name:<25} {float(map_disease[j]):.3f}')

### Pairplots

A **pairplot** shows the marginal posterior distributions for each parameter
(diagonal, as 1D KDE) and the joint distribution for each pair of parameters
(off-diagonal, as 2D KDE). The red dot marks the MAP estimate.

**What to look for:**
- **Width** of the marginals: wide = uncertain, narrow = well-constrained.
- **Correlations** in the off-diagonal panels: an elongated 2D distribution
  means the two parameters are degenerate -- you can trade one off against
  the other and still fit the data equally well.
- **Shift between healthy and disease**: which parameters move, and by how much?

In [ ]:
fig, axes = analysis.pairplot(
    samples_healthy,
    diag='kde', upper='kde',
    limits=prior_limits, ticks=prior_limits,
    points=[map_healthy],
    points_colors=['#EF6F6C'],
    points_offdiag={'markersize': 8},
    figsize=(6, 6), labels=param_labels,
)
fig.suptitle('Posterior -- healthy recording  (red = MAP)', fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = analysis.pairplot(
    samples_disease,
    diag='kde', upper='kde',
    limits=prior_limits, ticks=prior_limits,
    points=[map_disease],
    points_colors=['#EF6F6C'],
    points_offdiag={'markersize': 8},
    figsize=(6, 6), labels=param_labels,
)
fig.suptitle('Posterior -- KCNQ2 recording  (red = MAP)', fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

**Questions:**
- How wide are the posteriors? What does this tell you about parameter identifiability?
- Which parameters shift most between healthy and disease?
  Is this consistent with what you know about KCNQ2?
- Are there correlations between parameters in the posterior?
  What do these tell you about degeneracy in the parameter space?

---
## Exercise A: how does feature choice affect inference?

The NDE was trained on all six features. But are all six equally useful?

In this exercise you select a **subset of features**, re-train the NDE on
the same 1000 pre-computed simulations, and compare the resulting posterior
to the one above.

Re-training takes only a few seconds because the simulations themselves
do not need to be re-run -- we just re-use the stored parameter-feature pairs
with a different column selection.

Think about which features genuinely capture something about the network
dynamics, and which might not. Try different subsets and see what happens
to the posterior -- does it get wider, narrower, or shift?

In [ ]:
# Load pre-computed training data
# parameters : shape (N_sims, 3) -- the parameter sets used for each simulation
# features   : shape (N_sims, 6) -- the corresponding feature vectors
# valid      : shape (N_sims,)   -- False if the network was silent (no spikes)
parameters  = np.load('parameters.npy')
features_np = np.load('features.npy')
valid       = np.load('valid.npy')

parameters_valid = parameters[valid]
features_valid   = features_np[valid]

print(f'Training simulations available: {int(np.sum(valid))}')
print(f'Feature columns (in order): {feature_names}')

In [ ]:
# -- Change the feature subset here and re-run the cells below ---------------
# Select features by column index in features_np:
#   0 : mean_firing_rate
#   1 : n_active_units
#   2 : cv_firing_rate
#   3 : poprate_variance
#   4 : early_late_ratio
#   5 : mean_pairwise_corr

FEATURE_INDICES = [0, 1, 2, 3, 4, 5]   # <-- change this

selected_names = [feature_names[i] for i in FEATURE_INDICES]
print(f'Selected features: {selected_names}')

In [ ]:
from sbi.inference import NPE

# Set up a new prior -- same bounds as before
prior_sub = BoxUniform(
    low=torch.tensor(PRIOR_LOW,  dtype=torch.float32),
    high=torch.tensor(PRIOR_HIGH, dtype=torch.float32)
)

# Build training tensors using only the selected feature columns
theta_train   = torch.tensor(parameters_valid,                    dtype=torch.float32)
x_train       = torch.tensor(features_valid[:, FEATURE_INDICES], dtype=torch.float32)

# Build observation tensors with the same feature subset
x_healthy_sub = torch.tensor([features_healthy[f] for f in selected_names], dtype=torch.float32)
x_disease_sub = torch.tensor([features_disease[f] for f in selected_names], dtype=torch.float32)

# Train a new NPE on the selected features.
# NPE learns p(theta | x) from (theta, x) pairs.
# append_simulations() stores the data; train() fits the neural density estimator;
# build_posterior() wraps it into a posterior object we can sample from.
inference_sub = NPE(prior=prior_sub)
inference_sub.append_simulations(theta_train, x_train)
print('Training NPE on selected features...')
density_estimator_sub = inference_sub.train(show_train_summary=False)
posterior_sub = inference_sub.build_posterior(density_estimator_sub)
print('Done.')

In [ ]:
# Sample from the new posterior and show the pairplot
samples_disease_sub = posterior_sub.sample(
    (2000,), x=x_disease_sub, show_progress_bars=False
)
posterior_sub.set_default_x(x_disease_sub)
map_disease_sub = posterior_sub.map(show_progress_bars=False).squeeze()

print('MAP estimate (disease) with NDE trained on subpart of features:')
for j, name in builtins.enumerate(param_labels):
    print(f'  {name:<25} {float(map_disease_sub[j]):.3f}')

fig, axes = analysis.pairplot(
    samples_disease_sub,
    diag='kde', upper='kde',
    limits=prior_limits, ticks=prior_limits,
    points=[map_disease_sub],
    points_colors=['#EF6F6C'],
    points_offdiag={'markersize': 8},
    figsize=(6, 6), labels=param_labels,
)
fig.suptitle(
    f'Posterior -- KCNQ2  |  features: {selected_names}',
    fontsize=8, y=1.01
)
plt.tight_layout()
plt.show()

---
## Exercise B (optional): what do the features actually encode?

The posterior tells us which parameters are consistent with the observed
feature vector. But what does each feature actually *tell* the posterior
about the parameters?

In this exercise you manually modify individual features of the disease
recording and re-run inference. By changing one feature at a time you can
see which parameter each feature is most informative about.

For example:
- What happens to the posterior if you double `poprate_variance`
  (imagine a much more bursty recording)?
- What if you set `mean_pairwise_corr` to near 0
  (a completely asynchronous network)?
- What if you set a feature to a value that is outside the range seen
  in the training simulations?

Use the pre-trained posterior from Section 3 (all six features) for this,
so you are only changing the observation, not the model.

In [ ]:
# Start from the disease feature vector and modify individual values.
# Run this cell and the pairplot cell below after each change.

# Copy the disease features so we do not overwrite them
features_perturbed = dict(features_disease)

# ── Modify one or more features here ─────────────────────────────────────────
features_perturbed['mean_pairwise_corr'] = 0.1   # <-- change this
# features_perturbed['poprate_variance']  = features_disease['poprate_variance'] * 2
# features_perturbed['mean_firing_rate']  = 5.0

# Show what changed
print(f'  {"Feature":<25}  {"Original":>10}  {"Perturbed":>10}')
print(f'  {"-"*25}  {"-"*10}  {"-"*10}')
for name in feature_names:
    orig = features_disease[name]
    pert = features_perturbed[name]
    marker = '  <--' if abs(orig - pert) > 1e-6 else ''
    print(f'  {name:<25}  {orig:>10.3f}  {pert:>10.3f}{marker}')

In [ ]:
# Run inference on the perturbed feature vector
x_perturbed = torch.tensor(
    [features_perturbed[f] for f in feature_names], dtype=torch.float32
)

samples_perturbed = posterior.sample((2000,), x=x_perturbed, show_progress_bars=False)
posterior.set_default_x(x_perturbed)
map_perturbed = posterior.map(show_progress_bars=False).squeeze()

print('MAP estimate (perturbed):')
for j, name in builtins.enumerate(param_labels):
    orig_val = float(map_disease[j])
    pert_val = float(map_perturbed[j])
    print(f'  {name:<25} {orig_val:.3f}  -->  {pert_val:.3f}')

In [ ]:
# Pairplot -- perturbed features vs original disease posterior
# Both posteriors are shown together for direct comparison.
fig, axes = analysis.pairplot(
    [samples_disease, samples_perturbed],
    diag='kde', upper='kde',
    limits=prior_limits, ticks=prior_limits,
    points=[map_disease, map_perturbed],
    points_colors=['#6B0504', '#EF6F6C'],
    points_offdiag={'markersize': 8},
    figsize=(6, 6), labels=param_labels,
)
fig.suptitle(
    'Dark red = original disease posterior,  red = perturbed features',
    fontsize=8, y=1.01
)
plt.tight_layout()
plt.show()

---
## Optional: train your own posterior with a custom feature

If you defined a custom feature in notebook 1, you can include it here.
Load the training spike trains, compute your feature for each simulation,
add it as a new column to the training features, and re-train.

The full training spike trains are available from Google Drive: https://drive.google.com/drive/folders/1ksxhRjNpyIfcXouNJezmrVgdfNmjBUCH?usp=sharing.

In [ ]:
# Download training spike trains (large -- uncomment when needed)

# import zipfile
# gdown.download_folder(id='1ksxhRjNpyIfcXouNJezmrVgdfNmjBUCH', output='spike_trains/', quiet=False)
# with zipfile.ZipFile('spike_trains.zip', 'r') as z:
#     z.extractall('spike_trains/')

In [ ]:
# Skeleton for adding a custom feature to the training data

# def my_feature(spike_data, sim_duration):
#     # ... your code here ...
#     return value
#
# my_features = np.zeros(int(np.sum(valid)))
# valid_indices = np.where(valid)[0]
# for k, sim_idx in builtins.enumerate(valid_indices):
#     sd = np.loadtxt(f'spike_trains/sim_{sim_idx:04d}.csv', delimiter=',', skiprows=1)
#     my_features[k] = my_feature(sd, 60.0)
#
# features_extended = np.column_stack([features_valid, my_features])
pass